In [2]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset
from PIL import Image
from torch.nn.utils.rnn import pad_sequence
from collections import Counter
import re
import random

random.seed(42)

img_dir = "/kaggle/input/datasets/quangnguyen711/flickr30k-images"
comp_dir = "/kaggle/input/datasets/quangnguyen711/flickr30k-label-data" 

df = pd.read_csv(os.path.join(comp_dir, "train.csv"))

df.columns = df.columns.str.strip()
df["image_name"] = df["image_name"].str.strip()
df = df.dropna(subset=["comment"]).reset_index(drop=True)

# Chia tập Train/Val
images = df["image_name"].unique()
train_imgs, val_imgs = train_test_split(images, test_size=0.1, random_state=42)

train_df = df[df["image_name"].isin(train_imgs)].reset_index(drop=True)
val_df   = df[df["image_name"].isin(val_imgs)].reset_index(drop=True)

test_df = pd.read_csv(os.path.join(comp_dir, "test.csv"))
test_df["image_name"] = test_df["image_name"].str.strip() 

def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9 ]", "", text)
    return text.split()

counter = Counter()
for caption in train_df["comment"]:
    counter.update(tokenize(caption))

vocab = ["<pad>", "<start>", "<end>", "<unk>"]
min_freq = 5
vocab += [w for w, f in counter.items() if f >= min_freq]

stoi = {w:i for i,w in enumerate(vocab)}
itos = {i:w for w,i in stoi.items()}

class FlickrDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def numericalize(self, text):
        return [stoi.get(t, stoi["<unk>"]) for t in tokenize(text)]

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row["image_name"])

        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)

        caption = [stoi["<start>"]]
        caption += self.numericalize(row["comment"])
        caption.append(stoi["<end>"])

        return image, torch.tensor(caption)

def collate_fn(batch):
    imgs, caps = zip(*batch)
    imgs = torch.stack(imgs)
    caps = pad_sequence(caps, batch_first=True, padding_value=stoi["<pad>"])
    return imgs, caps

print(f"Train size: {len(train_df)}")
print(f"Val size: {len(val_df)}")
print(f"Test size: {len(test_df)}")

Train size: 114414
Val size: 12715
Test size: 6357


In [3]:
import torchvision.models as models
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"

class EncoderCNN(nn.Module):
    def __init__(self, embed_size):
        super().__init__()
        resnet = models.resnet50(weights=None)
        
        state_dict = torch.load(
            "/kaggle/input/datasets/quangnguyen711/cnn-backbone-pytorch-weights/resnet50.pth",
            map_location="cpu"
        )
        resnet.load_state_dict(state_dict)
        
        for p in resnet.parameters():
            p.requires_grad = False

        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool

        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

        self.avgpool = resnet.avgpool
        
        self.fc = nn.Sequential(
            nn.Dropout(0.15),
            nn.Linear(resnet.fc.in_features, embed_size)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)

        return self.fc(x)

class DecoderRNN(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
        self.init_h = nn.Linear(embed_size, hidden_size)
        self.init_c = nn.Linear(embed_size, hidden_size)

    def forward(self, features, captions):
        emb = self.embedding(captions[:, :-1])
        h0 = self.init_h(features).unsqueeze(0)
        c0 = self.init_c(features).unsqueeze(0)
        out, _ = self.lstm(emb, (h0, c0))
        return self.fc(out)

class CaptionModel(nn.Module):
    def __init__(self, embed, hidden, vocab):
        super().__init__()
        self.encoder = EncoderCNN(embed)
        self.decoder = DecoderRNN(embed, hidden, vocab)

    def forward(self, img, cap):
        f = self.encoder(img)
        return self.decoder(f, cap)

In [4]:
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

train_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_loader = DataLoader(
    FlickrDataset(train_df, img_dir, train_tf),
    batch_size=128, shuffle=True, num_workers=32, collate_fn=collate_fn
)

val_loader = DataLoader(
    FlickrDataset(val_df, img_dir, val_tf),
    batch_size=128, shuffle=False, num_workers=32, collate_fn=collate_fn
)

In [5]:
def generate(model, img, max_len=20):
    model.eval()
    if img.dim() == 3:
        img = img.unsqueeze(0).to(device)

    with torch.no_grad():
        feat = model.encoder(img)

        h = model.decoder.init_h(feat).unsqueeze(0)
        c = (
            model.decoder.init_c(feat).unsqueeze(0)
            if hasattr(model.decoder, "init_c")
            else torch.zeros_like(h)
        )

        word = torch.tensor([[stoi["<start>"]]], device=device)
        result = []

        for _ in range(max_len):
            emb = model.decoder.embedding(word)

            out, (h, c) = model.decoder.lstm(emb, (h, c))
            logits = model.decoder.fc(out[:, -1, :])

            next_word = logits.argmax(1).item()
            result.append(next_word)

            if next_word == stoi["<end>"]:
                break

            word = torch.tensor([[next_word]], device=device)

    return [itos[i] for i in result]

In [6]:
from collections import defaultdict
from nltk.translate.bleu_score import corpus_bleu
import random
from tqdm import tqdm

img2caps = defaultdict(list)
for _, r in val_df.iterrows():
    img2caps[r["image_name"]].append(tokenize(r["comment"]))

def eval_bleu(model, dataset, n):
    model.eval()
    refs, hyps = [], []
    indices = random.sample(range(len(dataset)), min(n, len(dataset)))

    for i in tqdm(indices, desc="Evaluating BLEU"):
        img, _ = dataset[i]
        pred = generate(model, img)
        pred = [w for w in pred if w not in ["<start>","<end>","<pad>"]]

        img_name = dataset.df.iloc[i]["image_name"]
        gt = img2caps[img_name]

        refs.append(gt)
        hyps.append(pred)

    return {
        "BLEU1": corpus_bleu(refs, hyps, weights=(1,0,0,0)),
        "BLEU4": corpus_bleu(refs, hyps, weights=(0.25,)*4)
    }

In [7]:
import torch.optim as optim

model = CaptionModel(256, 512, len(vocab)).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=stoi["<pad>"])
optimizer = optim.AdamW(
    [
        {
            "params": model.encoder.fc.parameters(),
            "lr": 1e-4
        },
        {
            "params": model.decoder.parameters(),
            "lr": 1e-4
        }
    ],
    weight_decay=1e-4
)



scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

stage1_epochs = 10

best_val_loss = float("inf")
patience = 5
patience_counter = 0

for epoch in range(stage1_epochs):

    # ========================
    # TRAIN
    # ========================

    model.train()

    # Giữ CNN pretrained ở eval mode
    model.encoder.conv1.eval()
    model.encoder.bn1.eval()
    model.encoder.layer1.eval()
    model.encoder.layer2.eval()
    model.encoder.layer3.eval()
    model.encoder.layer4.eval()

    train_loss = 0

    for imgs, caps in tqdm(
        train_loader,
        desc=f"Stage 1 - Epoch {epoch} Train"
    ):

        imgs = imgs.to(device)
        caps = caps.to(device)

        optimizer.zero_grad()

        out = model(imgs, caps)

        loss = criterion(
            out.reshape(-1, out.shape[-1]),
            caps[:, 1:].reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ========================
    # VALIDATION
    # ========================

    model.eval()

    val_loss = 0

    with torch.no_grad():

        for imgs, caps in tqdm(
            val_loader,
            desc=f"Stage 1 - Epoch {epoch} Val"
        ):

            imgs = imgs.to(device)
            caps = caps.to(device)

            out = model(imgs, caps)

            loss = criterion(
                out.reshape(-1, out.shape[-1]),
                caps[:, 1:].reshape(-1)
            )

            val_loss += loss.item()

    val_loss /= len(val_loader)

    scheduler.step(val_loss)

    bleu = eval_bleu(
        model,
        FlickrDataset(val_df, img_dir, val_tf),
        n=200
    )

    print(
        f"\nStage 1 Epoch {epoch} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

    print(
        f"BLEU-1: {bleu['BLEU1']:.4f} | "
        f"BLEU-4: {bleu['BLEU4']:.4f}"
    )

    if val_loss < best_val_loss - 1e-4:

        best_val_loss = val_loss
        patience_counter = 0

        torch.save(
            model.state_dict(),
            "best_stage1.pth"
        )

    else:

        patience_counter += 1

        if patience_counter >= patience:
            print("Stage 1 early stopping")
            break

Evaluating BLEU: 100%|██████████| 200/200 [00:01<00:00, 104.48it/s]



Stage 1 Epoch 0 | Train Loss: 4.8888 | Val Loss: 4.2295
BLEU-1: 0.5335 | BLEU-4: 0.1176


Evaluating BLEU: 100%|██████████| 200/200 [00:01<00:00, 128.37it/s]



Stage 1 Epoch 1 | Train Loss: 4.0450 | Val Loss: 3.8460
BLEU-1: 0.5646 | BLEU-4: 0.1426


Evaluating BLEU: 100%|██████████| 200/200 [00:01<00:00, 136.33it/s]


...


Stage 1 Epoch 7 | Train Loss: 3.1305 | Val Loss: 3.1719
BLEU-1: 0.5549 | BLEU-4: 0.1631


Evaluating BLEU: 100%|██████████| 200/200 [00:01<00:00, 120.70it/s]



Stage 1 Epoch 8 | Train Loss: 3.0691 | Val Loss: 3.1308
BLEU-1: 0.5520 | BLEU-4: 0.1656


Evaluating BLEU: 100%|██████████| 200/200 [00:01<00:00, 120.57it/s]



Stage 1 Epoch 9 | Train Loss: 3.0159 | Val Loss: 3.0964
BLEU-1: 0.5529 | BLEU-4: 0.1547


In [8]:
model.load_state_dict(
    torch.load(
        "best_stage1.pth",
        map_location=device
    )
)

for p in model.encoder.layer4.parameters():
    p.requires_grad = True

optimizer = optim.AdamW(
    [
        {
            "params": model.encoder.layer4.parameters(),
            "lr": 1e-5
        },
        {
            "params": model.encoder.fc.parameters(),
            "lr": 1e-4
        },
        {
            "params": model.decoder.parameters(),
            "lr": 1e-4
        }
    ],
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

stage2_epochs = 20

best_val_loss = float("inf")
patience = 7
patience_counter = 0

for epoch in range(stage2_epochs):

    # ========================
    # TRAIN
    # ========================

    model.train()

    # Các layer frozen giữ eval
    model.encoder.conv1.eval()
    model.encoder.bn1.eval()

    model.encoder.layer1.eval()
    model.encoder.layer2.eval()
    model.encoder.layer3.eval()

    # layer4 vẫn để eval để giữ BatchNorm running statistics
    # Conv weight vẫn được train vì requires_grad=True
    model.encoder.layer4.eval()

    train_loss = 0

    for imgs, caps in tqdm(
        train_loader,
        desc=f"Stage 2 - Epoch {epoch} Train"
    ):

        imgs = imgs.to(device)
        caps = caps.to(device)

        optimizer.zero_grad()

        out = model(imgs, caps)

        loss = criterion(
            out.reshape(-1, out.shape[-1]),
            caps[:, 1:].reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ========================
    # VALIDATION
    # ========================

    model.eval()

    val_loss = 0

    with torch.no_grad():

        for imgs, caps in tqdm(
            val_loader,
            desc=f"Stage 2 - Epoch {epoch} Val"
        ):

            imgs = imgs.to(device)
            caps = caps.to(device)

            out = model(imgs, caps)

            loss = criterion(
                out.reshape(-1, out.shape[-1]),
                caps[:, 1:].reshape(-1)
            )

            val_loss += loss.item()

    val_loss /= len(val_loader)

    scheduler.step(val_loss)

    bleu = eval_bleu(
        model,
        FlickrDataset(val_df, img_dir, val_tf),
        n=200
    )

    print(
        f"\nStage 2 Epoch {epoch} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

    print(
        f"BLEU-1: {bleu['BLEU1']:.4f} | "
        f"BLEU-4: {bleu['BLEU4']:.4f}"
    )

    if val_loss < best_val_loss - 1e-4:

        best_val_loss = val_loss
        patience_counter = 0

        torch.save(
            model.state_dict(),
            "best_model.pth"
        )

    else:

        patience_counter += 1

        if patience_counter >= patience:
            print("Stage 2 early stopping")
            break

Evaluating BLEU: 100%|██████████| 200/200 [00:01<00:00, 136.09it/s]



Stage 2 Epoch 0 | Train Loss: 2.9733 | Val Loss: 3.0619
BLEU-1: 0.5976 | BLEU-4: 0.1863


Evaluating BLEU: 100%|██████████| 200/200 [00:01<00:00, 136.68it/s]



Stage 2 Epoch 1 | Train Loss: 2.9125 | Val Loss: 3.0298
BLEU-1: 0.6231 | BLEU-4: 0.2254


Evaluating BLEU: 100%|██████████| 200/200 [00:01<00:00, 120.40it/s]


...


Stage 2 Epoch 17 | Train Loss: 2.3908 | Val Loss: 2.8766
BLEU-1: 0.6386 | BLEU-4: 0.2205


Evaluating BLEU: 100%|██████████| 200/200 [00:01<00:00, 133.42it/s]



Stage 2 Epoch 18 | Train Loss: 2.3684 | Val Loss: 2.8755
BLEU-1: 0.6056 | BLEU-4: 0.1951


Evaluating BLEU: 100%|██████████| 200/200 [00:01<00:00, 137.17it/s]



Stage 2 Epoch 19 | Train Loss: 2.3457 | Val Loss: 2.8763
BLEU-1: 0.6080 | BLEU-4: 0.1955


In [9]:
def beam_search(model, img_tensor, beam_width=5, max_len=20):
    model.eval()

    img_tensor = img_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        # 1. Encode image
        features = model.encoder(img_tensor)

        # 2. Initial hidden states từ image feature
        h0 = model.decoder.init_h(features).unsqueeze(0)
        c0 = model.decoder.init_c(features).unsqueeze(0)

        start_idx = stoi["<start>"]
        end_idx = stoi["<end>"]

        # (sequence, log_probability, hidden_state)
        sequences = [
            ([start_idx], 0.0, (h0, c0))
        ]

        for _ in range(max_len):

            all_candidates = []

            for seq, score, hidden in sequences:

                # Nếu đã gặp <end> thì giữ nguyên beam
                if seq[-1] == end_idx:
                    all_candidates.append(
                        (seq, score, hidden)
                    )
                    continue

                # Token cuối cùng
                input_token = torch.tensor(
                    [[seq[-1]]],
                    dtype=torch.long,
                    device=device
                )

                # Embedding
                emb = model.decoder.embedding(input_token)

                # chạy 1 step LSTM
                out, next_hidden = model.decoder.lstm(
                    emb,
                    hidden
                )

                # prediction
                logits = model.decoder.fc(
                    out[:, -1, :]
                )

                log_probs = torch.log_softmax(
                    logits,
                    dim=-1
                )

                # lấy beam_width token tốt nhất
                topk_log_probs, topk_indices = torch.topk(
                    log_probs,
                    beam_width,
                    dim=-1
                )

                for i in range(beam_width):

                    next_token = topk_indices[0, i].item()
                    next_score = topk_log_probs[0, i].item()

                    candidate = (
                        seq + [next_token],
                        score + next_score,
                        next_hidden
                    )

                    all_candidates.append(candidate)

            # Length normalization
            ordered = sorted(
                all_candidates,
                key=lambda x: x[1] / len(x[0]),
                reverse=True
            )

            sequences = ordered[:beam_width]

            # Nếu tất cả beam đã sinh <end>
            if all(
                seq[-1] == end_idx
                for seq, _, _ in sequences
            ):
                break

        # Best sequence
        best_seq = sequences[0][0]

        words = [
            itos[idx]
            for idx in best_seq
            if idx not in [
                stoi["<start>"],
                stoi["<end>"],
                stoi["<pad>"]
            ]
        ]

        return " ".join(words)

def generate_submission(
    model,
    test_df,
    img_dir,
    val_tf,
    output_file="submission.csv",
    beam_width=5,
    max_len=20,
):
    print("\n=== GENERATING SUBMISSION FOR KAGGLE ===")

    model.load_state_dict(
        torch.load(
            "best_model.pth",
            map_location=device
        )
    )

    model.eval()

    test_images = test_df["image_name"].unique()
    results = []

    for img_name in tqdm(
        test_images,
        desc="Predicting Test Set"
    ):

        img_path = os.path.join(
            img_dir,
            img_name
        )

        try:
            img = Image.open(
                img_path
            ).convert("RGB")

            img_tensor = val_tf(img)

            caption = beam_search(
                model,
                img_tensor,
                beam_width=beam_width,
                max_len=max_len
            )

        except Exception as e:
            print(
                f"Lỗi ảnh {img_name}: {e}"
            )
            caption = "a picture of something"

        results.append({
            "image_name": img_name,
            "comment": caption
        })

    sub_df = pd.DataFrame(results)

    sub_df.to_csv(
        output_file,
        index=False
    )

    print(
        f"Submission saved to {output_file}"
    )
generate_submission(model, test_df, img_dir, val_tf, beam_width=5)


=== GENERATING SUBMISSION FOR KAGGLE ===


Predicting Test Set: 100%|██████████| 6357/6357 [02:55<00:00, 36.27it/s]

Submission saved to submission.csv
